<a href="https://colab.research.google.com/github/eunmindow-star/chaeeunmin/blob/main/11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. 구글 드라이브 연결 및 원시자료 로드
drive.mount('/content/drive')
df = pd.read_sas('/content/drive/MyDrive/hdac_data/kyrbs2025.sas7bdat')

# 2. 노출(bf_skip) 및 결과(obesity) 생성 (10주차와 동일)
df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)
df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
                         np.where(df['F_BR'].notna(), 0, np.nan))
df['bmi'] = df['WT'] / ((df['HT'] / 100) ** 2)
df['obesity'] = np.where(df['bmi'] >= 25, 1,
                         np.where(df['bmi'].notna(), 0, np.nan))

# 3. 교란변수(SEX, GRADE, E_SES) 결측 처리
df['sex_bin'] = np.where(df['SEX'] == 1, 1, np.where(df['SEX'] == 2, 0, np.nan))
ses_col = 'E_SES' if 'E_SES' in df.columns else 'PR_INC'
df[ses_col] = df[ses_col].replace(9, np.nan)

# 4. 모형 1(무보정)과 모형 2(보정)를 '같은 사람'으로 맞추기 위한 공통 유효 도메인 정의
model_vars = ['bf_skip', 'obesity', 'sex_bin', 'GRADE', ses_col]
df['in_model12'] = ((df['domain_hs'] == 1) & df[model_vars].notna().all(axis=1)).astype(int)

print(f"전체 조사 표본: {len(df):,}명")
print(f"10주차 유효 분석 대상자: {((df['domain_hs'] == 1) & df[['bf_skip', 'obesity']].notna().all(axis=1)).sum():,}명")
print(f"11주차 모형 1·2 공통 유효 대상자(같은 사람): {df['in_model12'].sum():,}명")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
전체 조사 표본: 54,170명
10주차 유효 분석 대상자: 25,077명
11주차 모형 1·2 공통 유효 대상자(같은 사람): 25,070명


/tmp/ipykernel_1949/1020160141.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)
/tmp/ipykernel_1949/1020160141.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
/tmp/ipykernel_1949/1020160141.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instea

In [2]:
# 모형에 포함될 변수들 중 결측치가 없는 유효 데이터만 필터링 (공통 N = 25,070명)
analysis_df = df[df['in_model12'] == 1].copy()

# 층(STRATA)-집락(CLUSTER) 결합 PSU ID 생성
analysis_df['psu_id'] = analysis_df['STRATA'].astype(str) + "_" + analysis_df['CLUSTER'].astype(str)

# ★ 가중치 정규화 (평균이 1이 되도록 스케일링하여 표준오차 정확도 향상)
analysis_df['w_norm'] = analysis_df['W'] / analysis_df['W'].mean()

# 모형 1: 무보정 모형 (Unadjusted) -> freq_weights에 w_norm 사용
m1 = smf.glm(
    'obesity ~ bf_skip',
    data=analysis_df,
    family=sm.families.Binomial(),
    freq_weights=analysis_df['w_norm']
).fit(cov_type='cluster', cov_kwds={'groups': analysis_df['psu_id']})

# 모형 2: 최소충분보정집합 보정 모형 (Adjusted: 성별, 학년, 주관적 경제상태) -> freq_weights에 w_norm 사용
formula_m2 = f'obesity ~ bf_skip + C(sex_bin) + C(GRADE) + C({ses_col})'
m2 = smf.glm(
    formula_m2,
    data=analysis_df,
    family=sm.families.Binomial(),
    freq_weights=analysis_df['w_norm']
).fit(cov_type='cluster', cov_kwds={'groups': analysis_df['psu_id']})

# OR 및 95% 신뢰구간 산출 함수
def get_or_ci(model, var_name='bf_skip'):
    beta = model.params[var_name]
    se = model.bse[var_name]
    pval = model.pvalues[var_name]
    or_val = np.exp(beta)
    ci_low = np.exp(beta - 1.96 * se)
    ci_high = np.exp(beta + 1.96 * se)
    return or_val, ci_low, ci_high, pval

or1, l1, h1, p1 = get_or_ci(m1)
or2, l2, h2, p2 = get_or_ci(m2)

n_common = int(analysis_df['in_model12'].sum())
n_skip = int(((analysis_df['in_model12'] == 1) & (analysis_df['bf_skip'] == 1)).sum())
n_nonskip = int(((analysis_df['in_model12'] == 1) & (analysis_df['bf_skip'] == 0)).sum())

table2_df = pd.DataFrame([
    {
        '노출 구분': '아침식사 비결식군 (주 0–4회, Ref)',
        '분석 인원 (n)': f"{n_nonskip:,}",
        'Model 1: Unadjusted OR (95% CI)': '1.00 (Reference)',
        'Model 1 p-value': '-',
        'Model 2: Adjusted OR (95% CI)': '1.00 (Reference)',
        'Model 2 p-value': '-'
    },
    {
        '노출 구분': '아침식사 결식군 (주 5회 이상)',
        '분석 인원 (n)': f"{n_skip:,}",
        'Model 1: Unadjusted OR (95% CI)': f"{or1:.2f} ({l1:.2f}–{h1:.2f})",
        'Model 1 p-value': f"{p1:.4f}",
        'Model 2: Adjusted OR (95% CI)': f"{or2:.2f} ({l2:.2f}–{h2:.2f})",
        'Model 2 p-value': f"{p2:.4f}"
    }
])

print(f"=== [C-2] Table 2: 아침식사 결식과 비만 유병의 설계 기반 로지스틱 회귀분석 (공통 N = {n_common:,}명) ===")
display(table2_df)

/usr/local/lib/python3.13/dist-packages/statsmodels/genmod/generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(


=== [C-2] Table 2: 아침식사 결식과 비만 유병의 설계 기반 로지스틱 회귀분석 (공통 N = 25,070명) ===


/usr/local/lib/python3.13/dist-packages/statsmodels/genmod/generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(


,노출 구분,분석 인원 (n),Model 1: Unadjusted OR (95% CI),Model 1 p-value,Model 2: Adjusted OR (95% CI),Model 2 p-value
0,"아침식사 비결식군 (주 0–4회, Ref)","13,695",1.00 (Reference),-,1.00 (Reference),-
1,아침식사 결식군 (주 5회 이상),"11,375",1.07 (1.00–1.15),0.0564,1.09 (1.01–1.17),0.0211
